# NewsAPI

### Set Up

In [36]:
import os
from dotenv import load_dotenv, dotenv_values
from pathlib import Path

In [37]:
load_dotenv(Path.cwd().parent / ".env")

True

In [38]:
API_KEY_NEWS = os.getenv("API_KEY_NEWS")

print(API_KEY_NEWS)

5f1946f04b7642ab944af4806cdc7e4b


## EDA

In [ ]:
"""
NewsAPI EDA — Single Company
=============================
Pulls articles for one company and runs a full EDA:
  - Shape, columns, dtypes
  - Missing values
  - Duplicate articles
  - Articles over time
  - Source distribution
  - Article text length
  - Most common words

Install:
    pip install requests pandas numpy matplotlib seaborn

Run:
    python newsapi_eda.py
"""

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from collections import Counter

# ── Config ────────────────────────────────────────────────────────────
COMPANY_NAME = "barnsley"

sns.set_theme(style="whitegrid", palette="muted")

pd.set_option("display.max_columns",  None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width",        140)



# 1. PULL DATA

def pull_news(company_name, days_back=28, page_size=100, page=1):
    clean = (company_name
             .replace(" PLC", "").replace(" plc", "")
             .replace(" Ltd", "").replace(" Limited", "")
             .strip())

    from_date = (datetime.now() - timedelta(days=days_back)).strftime("%Y-%m-%d")

    response = requests.get("https://newsapi.org/v2/everything", params={
        "q":        f'"{clean}"',
        "language": "en",
        "sortBy":   "publishedAt",
        "pageSize": page_size,
        "page":     page,
        "from":     from_date,
        "apiKey":   API_KEY_NEWS,
    })

    data = response.json()
    articles = data.get("articles", [])

    print(f"API status     : {data.get('status')}")
    print(f"Page requested : {page}")
    print(f"Total available: {data.get('totalResults')}")
    print(f"Articles returned this call: {len(articles)}")
    print(f"Error code     : {data.get('code')}")
    print(f"Error message  : {data.get('message')}")

    df = pd.json_normalize(articles)
    return df, data.get("totalResults")


print("="*60)
print(f"PULLING NEWS FOR: {COMPANY_NAME}")
print("="*60)
df, total_results = pull_news(COMPANY_NAME, page=1)

if df.empty:
    print("\n⚠ No data returned — stopping here.")
    print("Check the error message above and fix it before continuing.")
    raise SystemExit

print(f"\nTOTAL NEWS AVAILABLE FOR '{COMPANY_NAME}': {total_results}")
print(f"TOTAL NEWS RETRIEVED (this run)         : {len(df)}")



# 2. SHAPE, COLUMNS, DTYPES

print("\n" + "="*60)
print("2. SHAPE, COLUMNS, DTYPES")
print("="*60)

print(f"\nShape   : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns and types:")
print(df.dtypes)

print(f"\nhead:")
#print(df[["publishedAt", "source.name", "title"]].head(5).to_string(index=False))
print(df.head)


# 3. MISSING VALUES

print("\n" + "="*60)
print("3. MISSING VALUES")
print("="*60)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print(missing_df[missing_df["missing_count"] > 0].to_string())

if missing_df["missing_count"].sum() == 0:
    print("No missing values found.")



# 4. DUPLICATE ARTICLES

print("\n" + "="*60)
print("4. DUPLICATE ARTICLES")
print("="*60)

dupes_by_title = df.duplicated(subset=["title"]).sum()
dupes_by_url   = df.duplicated(subset=["url"]).sum()

print(f"Duplicate titles : {dupes_by_title}")
print(f"Duplicate URLs   : {dupes_by_url}")

if dupes_by_title > 0:
    print("\nExample duplicates:")
    dupe_titles = df[df.duplicated(subset=["title"], keep=False)]
    print(dupe_titles[["publishedAt", "source.name", "title"]].head(4).to_string(index=False))

# Drop duplicates for the rest of the analysis
df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)
print(f"\nRows after removing duplicates: {len(df)}")



# 5. ARTICLES OVER TIME

print("\n" + "="*60)
print("5. ARTICLES OVER TIME")
print("="*60)

df["publishedAt"] = pd.to_datetime(df["publishedAt"])
df["date"] = df["publishedAt"].dt.date

daily_counts = df.groupby("date").size().reset_index(name="count")
print(daily_counts.to_string(index=False))

plt.figure(figsize=(10, 4))
plt.plot(daily_counts["date"], daily_counts["count"], marker="o", color="#4C72B0")
plt.fill_between(daily_counts["date"], daily_counts["count"], alpha=0.2, color="#4C72B0")
plt.title(f"Articles per day — {COMPANY_NAME}")
plt.xlabel("Date")
plt.ylabel("Article count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("newsapi_eda_timeline.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nChart saved → newsapi_eda_timeline.png")



# 6. SOURCE DISTRIBUTION

print("\n" + "="*60)
print("6. SOURCE DISTRIBUTION")
print("="*60)

source_counts = df["source.name"].value_counts()
print(source_counts.to_string())

plt.figure(figsize=(8, max(3, len(source_counts) * 0.4)))
source_counts.sort_values().plot(kind="barh", color="#55A868")
plt.title(f"Articles by source — {COMPANY_NAME}")
plt.xlabel("Article count")
plt.tight_layout()
plt.savefig("newsapi_eda_sources.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nChart saved → newsapi_eda_sources.png")



# 7. ARTICLE TEXT LENGTH

print("\n" + "="*60)
print("7. ARTICLE TEXT LENGTH")
print("="*60)

df["title_len"]       = df["title"].fillna("").apply(len)
df["description_len"] = df["description"].fillna("").apply(len)

print("\nTitle length (characters):")
print(df["title_len"].describe().round(1))

print("\nDescription length (characters):")
print(df["description_len"].describe().round(1))

empty_desc = (df["description"].fillna("") == "").sum()
print(f"\nArticles with empty description: {empty_desc} / {len(df)}")



# 8. MOST COMMON WORDS IN TITLES

print("\n" + "="*60)
print("8. MOST COMMON WORDS IN TITLES")
print("="*60)

STOPWORDS = set("""
the a an and or but in on at to for of with by from is are was were be been
this that these those it its as has have had will would could should
""".split())

all_words = []
for title in df["title"].fillna(""):
    words = [w.lower().strip(".,!?:;'\"()") for w in title.split()]
    words = [w for w in words if w and w not in STOPWORDS and len(w) > 2]
    all_words.extend(words)

word_counts = Counter(all_words)
print("\nTop 15 words:")
for word, count in word_counts.most_common(15):
    print(f"  {word:<20} {count}")



# 9. SAVE CLEANED DATA


print("\n" + "="*60)
print("9. SAVE CLEANED DATA")
print("="*60)

output_cols = ["publishedAt", "date", "source.name", "title", "description",
               "title_len", "description_len", "url"]
df[output_cols].to_csv(f"newsapi_eda_{COMPANY_NAME.lower()}.csv", index=False)
print(f"\nSaved → newsapi_eda_{COMPANY_NAME.lower()}.csv")

print("\n" + "="*60)
print("DONE — files saved:")
print("  newsapi_eda_timeline.png")
print("  newsapi_eda_sources.png")
print(f"  newsapi_eda_{COMPANY_NAME.lower()}.csv")
print("="*60)

PULLING NEWS FOR: barnsley
API status     : ok
Page requested : 1
Total available: 77
Articles returned this call: 71
Error code     : None
Error message  : None

TOTAL NEWS AVAILABLE FOR 'barnsley': 77
TOTAL NEWS RETRIEVED (this run)         : 71

2. SHAPE, COLUMNS, DTYPES

Shape   : 71 rows x 9 columns

Columns and types:
author         object
title          object
description    object
url            object
urlToImage     object
publishedAt    object
content        object
source.id      object
source.name    object
dtype: object

head:
<bound method NDFrame.head of                                             author                                                        title  \
0   Emlyn Begley - BBC Sport journalist in Houston  Smaller than Isle of Man & huge Dutch influence: Curacao...   
1                                      Mark Palmer  Curacao prepare to take on Germany! The World Cup's smal...   
2                                        Liz Ivens  Defence Secretary John Heale

EDA - results
- Many unrelated articles --> Need to be filtered
- Some duplications --> Clean
- Some columns are not useful --> Clean
- Limitation for Free Tier --> 1-month old only --> Even Tesco has only a few data --> SME will be worse
- 100 requests per day --> 100 companies limit per day


What we can get from NewsAPI
- Main source for Media Coverage (Unstructered Data) to be linked with Companies House Data (Structured)
- Provide Trends and Correletions with Companies House data 
- Add predictive power to the model (detect patterns in external behaviour and news coverage)

Questions
- How should we select companies?